# Causal Multi-Head Self-Attention (MHA)
## Chapter 3: Scalable Multi-Head Parallelism for Transformers

Single-head self-attention computes an average of relationships, which can dilute distinct linguistic features (e.g., syntactic structure, coreference, factual relations). 

**Multi-Head Attention (MHA)** solves this by partitioning the embedding dimension $d_{\text{model}}$ into $h$ distinct subspace heads, each running scaled dot-product attention in parallel.

---

### Architectural Formulation:
Given $h$ heads and total hidden dimension $d_{\text{model}}$:
- **Head Dimension**: $d_{\text{head}} = d_{\text{model}} / h$

1. **Unified Linear Projections**:
   Instead of $h$ separate linear layers, project inputs in parallel using single matrices $\mathbf{W}_Q, \mathbf{W}_K, \mathbf{W}_V \in \mathbb{R}^{d_{\text{model}} \times d_{\text{model}}}$.

2. **Tensor Reshaping & Head Splitting**:
   $$(B, T, d_{\text{model}}) \xrightarrow{\text{view}} (B, T, h, d_{\text{head}}) \xrightarrow{\text{transpose}(1, 2)} (B, h, T, d_{\text{head}})$$

3. **Batched Scaled Dot-Product with Causal Mask**:
   $$\mathbf{S} = \frac{\mathbf{Q} \mathbf{K}^\top}{\sqrt{d_{\text{head}}}}, \quad \text{causal mask on future tokens (}\text{triu}(..., \text{diagonal}=1)\text{)}$$
   $$\mathbf{A} = \text{softmax}(\mathbf{S}_{\text{masked}}, \text{dim}=-1)$$

4. **Multi-Head Aggregation & Projection**:
   $$\mathbf{Z} = \mathbf{A} \mathbf{V} \in \mathbb{R}^{B \times h \times T \times d_{\text{head}}}$$
   $$(B, h, T, d_{\text{head}}) \xrightarrow{\text{transpose}(1, 2)} (B, T, h, d_{\text{head}}) \xrightarrow{\text{view}} (B, T, d_{\text{model}})$$
   $$\mathbf{Y} = \mathbf{Z}_{\text{concat}} \mathbf{W}_{O}$$
   where $\mathbf{W}_{O} \in \mathbb{R}^{d_{\text{model}} \times d_{\text{model}}}$ is the final linear output projection.

---
## 1. Imports and Dependencies

In [7]:
import torch 
import math 
import torch.nn as nn 
import torch.nn.functional as F

---
## 2. Implementing `CausalMultiHeadSelfAttention`

In this implementation:
- We project Queries, Keys, and Values using full-rank linear layers.
- Reshape and transpose tensors to compute all $h$ attention heads in parallel with batched matrix multiplication (`@`).
- Mask future tokens using `torch.triu(..., diagonal=1)` to enforce autoregressive causality.
- Concatenate head outputs and project through `out_proj`.

In [8]:
class CausalMultiHeadSelfAttention(nn.Module):

    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_query = nn.Linear(d_model, d_model, bias=False)
        self.W_key = nn.Linear(d_model, d_model, bias=False)
        self.W_value = nn.Linear(d_model, d_model, bias=False)

        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):

        batch_size, num_tokens, _ = x.shape

        Q = self.W_query(x)
        K = self.W_key(x)
        V = self.W_value(x)

        Q = Q.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1)
        scores = scores / (self.head_dim ** 0.5)

        causal_mask = torch.triu(torch.ones(num_tokens, num_tokens, device=x.device), diagonal=1).bool()

        scores = scores.masked_fill(causal_mask, float('-inf'))

        attention_weights = torch.softmax(scores, dim=-1)

        context = attention_weights @ V

        context = context.transpose(1, 2).contiguous().view(
            batch_size,
            num_tokens,
            self.d_model
        )
        
        output = self.out_proj(context)

        return output, attention_weights


---
## 3. Dimensional Analysis & Execution

We test `CausalMultiHeadSelfAttention` with typical GPT-style hyperparameters:
- **Batch Size ($B$)**: 2
- **Number of Tokens ($T$)**: 8
- **Model Dimension ($d_{\text{model}}$)**: 256
- **Number of Heads ($h$)**: 8 (yielding $d_{\text{head}} = 256 / 8 = 32$)

Expected shapes:
- Input: `[2, 8, 256]`
- Output: `[2, 8, 256]`
- Attention Weights: `[2, 8, 8, 8]` $\rightarrow$ `(batch_size, num_heads, num_tokens, num_tokens)`

In [9]:
batch_size = 2
num_tokens = 8
d_model = 256
num_heads = 8

x = torch.randn(batch_size, num_tokens, d_model)

attention = CausalMultiHeadSelfAttention(
    d_model=d_model,
    num_heads=num_heads
)

output, attention_weights = attention(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Attention shape:", attention_weights.shape)

Input shape: torch.Size([2, 8, 256])
Output shape: torch.Size([2, 8, 256])
Attention shape: torch.Size([2, 8, 8, 8])
